# 03: MERGE SEGMENTATION CLASSES

This notebook merges ADE20K PSPNet segmentation masks into 7 super-classes optimized for HCMC urban analysis: other, vegetation, sky, building, pavement/road, water, and vehicle/clutter.

## MODULE SETUP

In [ ]:
# Mount Google Drive.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

# Set working directory to project folder.
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/hot_hem"
os.chdir(BASE_DIR)

print(f"Working directory: {BASE_DIR}")

## IMPORT SETUP

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm import tqdm

## PATH CONFIGURATION

In [ ]:
# Input/output base directory.
IMAGE_DIR = Path("data/processing/images")

# Output metrics file.
OUT_DIR = Path("data/outputs/features")
OUT_CSV = OUT_DIR / "superclass_metrics.csv"

# Checkpoint file.
CHECKPOINT_FILE = Path("data/processing/gsv/merge_checkpoint.json")

# Create output directory.
OUT_DIR.mkdir(parents = True, exist_ok = True)

print("PATH CONFIGURATION")
print(f"Image directory: {IMAGE_DIR}")
print(f"Output metrics: {OUT_CSV}")
print(f"Checkpoint file: {CHECKPOINT_FILE}")

## DEFINE SUPER CLASSES

In [ ]:
# Dictionary mapping superclass names to integer labels.
SUPER_CLASSES = {
    "other": 0,
    "vegetation": 1,
    "sky": 2,
    "building": 3,
    "pavement_road": 4,
    "water": 5,
    "vehicle_clutter": 6,
}

print("SUPER CLASSES")
for name, label in SUPER_CLASSES.items():
    print(f"{label}: {name}")

In [ ]:
# ADE20K class IDs grouped into super-classes.

# Vegetation IDs: foliage, grass, trees.
VEGETATION_IDS = {5, 13, 20, 41, 56, 100, 129, 148}

# Sky IDs.
SKY_IDS = {3}

# Building IDs: structures, windows, doors, railings.
BUILDING_IDS = {1, 4, 6, 10, 28, 50, 59, 89, 90, 114, 115, 133}

# Pavement/Road IDs: asphalt, concrete, sidewalks.
PAVEMENT_ROAD_IDS = {2, 7, 18, 24, 32, 74, 75, 104, 111, 135}

# Water IDs.
WATER_IDS = {22, 55, 57, 93}

# Vehicle and Clutter IDs: motorcycles, cars, signs, poles.
# Critical for HCMC street scenes with heavy traffic.
VEHICLE_CLUTTER_IDS = {
    8,    # Pole.
    12,   # Sign, signboard.
    17,   # Car, auto.
    19,   # Bus.
    39,   # Traffic light.
    40,   # Truck, lorry.
    46,   # Motorcycle, scooter.
    70,   # Street lamp.
    71,   # Street light.
    72,   # Street sign.
    144,  # Traffic sign.
}

print("ADE20K CLASS MAPPINGS")
print(f"Vegetation: {len(VEGETATION_IDS)} classes")
print(f"Sky: {len(SKY_IDS)} classes")
print(f"Building: {len(BUILDING_IDS)} classes")
print(f"Pavement/Road: {len(PAVEMENT_ROAD_IDS)} classes")
print(f"Water: {len(WATER_IDS)} classes")
print(f"Vehicle/Clutter: {len(VEHICLE_CLUTTER_IDS)} classes")

In [ ]:
# Build lookup dictionary mapping ADE20K classes to super-classes.
lookup = {}

# Process all 151 ADE20K classes (0-150).
all_ids = set(range(151))

for cid in all_ids:
    if cid in VEGETATION_IDS:
        lookup[cid] = SUPER_CLASSES["vegetation"]
    elif cid in SKY_IDS:
        lookup[cid] = SUPER_CLASSES["sky"]
    elif cid in BUILDING_IDS:
        lookup[cid] = SUPER_CLASSES["building"]
    elif cid in PAVEMENT_ROAD_IDS:
        lookup[cid] = SUPER_CLASSES["pavement_road"]
    elif cid in WATER_IDS:
        lookup[cid] = SUPER_CLASSES["water"]
    elif cid in VEHICLE_CLUTTER_IDS:
        lookup[cid] = SUPER_CLASSES["vehicle_clutter"]
    else:
        lookup[cid] = SUPER_CLASSES["other"]

# Extend lookup to handle all uint8 values (0-255).
for i in range(256):
    if i not in lookup:
        lookup[i] = SUPER_CLASSES["other"]

print(f"Lookup mapping ready. Total super-classes: {len(SUPER_CLASSES)}")
print(f"Sample mappings: {dict(list(lookup.items())[:10])}")

## MERGE MASK HELPER FUNCTION

In [ ]:
def merge_mask(mask_array, lookup_dict):
    """
    Convert ADE20K label mask into superclass mask.
    Applies lookup mapping to each pixel using vectorization.
    Returns uint8 array with labels 0-6.
    """
    merged = np.vectorize(lookup_dict.get)(mask_array)
    return merged.astype(np.uint8)

## VERIFY MERGE LOGIC

In [ ]:
# Find sample mask to test merge logic.
sample_masks = list(IMAGE_DIR.rglob("segmented/class_*.png"))

if sample_masks:
    test_mask_path = sample_masks[0]
    sample_mask = np.array(Image.open(test_mask_path))
    
    print(f"Test mask: {test_mask_path}")
    print(f"Raw ADE20K IDs in sample: {np.unique(sample_mask)}")
    
    merged_sample = merge_mask(sample_mask, lookup)
    print(f"Superclass IDs in merged mask: {np.unique(merged_sample)}")
else:
    print("No segmented masks found for testing.")

## LOAD CHECKPOINT AND EXISTING METRICS

In [ ]:
# Load existing metrics if available.
if OUT_CSV.exists():
    df_existing = pd.read_csv(OUT_CSV)
    done_uids = set(df_existing["uid"].astype(str))
    records = df_existing.to_dict("records")
    print(f"Loaded {len(done_uids)} existing processed masks.")
else:
    done_uids = set()
    records = []
    print("No existing metrics found. Starting fresh.")

## PROCESS SEGMENTATION MASKS

In [ ]:
# Process all segmented masks.
new_records = []

# Walk through image directory structure.
for district_dir in IMAGE_DIR.iterdir():
    # Skip non-directories.
    if not district_dir.is_dir():
        continue
    
    # Skip unknown districts.
    if "nan" in district_dir.name.lower():
        continue
    
    for ward_dir in district_dir.iterdir():
        # Skip non-directories.
        if not ward_dir.is_dir():
            continue
        
        # Check for segmented folder.
        seg_dir = ward_dir / "segmented"
        if not seg_dir.exists():
            continue
        
        # Create superclass output folder.
        superclass_dir = ward_dir / "superclass"
        superclass_dir.mkdir(parents = True, exist_ok = True)
        
        # Get all segmented masks.
        png_files = list(seg_dir.glob("class_*.png"))
        
        if not png_files:
            continue
        
        rel_path = f"{district_dir.name}/{ward_dir.name}"
        print(f"Processing {rel_path} ({len(png_files)} masks).")
        
        for mask_file in tqdm(png_files, desc = rel_path):
            # Extract UID from filename (class_###.png -> ###).
            uid = mask_file.stem.replace("class_", "")
            
            # Skip if already processed.
            if uid in done_uids:
                continue
            
            # Load mask.
            mask = cv2.imread(str(mask_file), cv2.IMREAD_UNCHANGED)
            
            if mask is None:
                continue
            
            # Handle 3-channel PNG if present.
            if len(mask.shape) == 3:
                mask = mask[:, :, 0]
            
            mask = mask.astype(np.uint8)
            
            # Merge to superclass.
            merged = merge_mask(mask, lookup)
            
            # Save merged mask with superclass_ prefix.
            out_path = superclass_dir / f"superclass_{uid}.png"
            Image.fromarray(merged).save(out_path)
            
            # Calculate pixel metrics.
            total = merged.size
            
            rec = {
                "uid": uid,
                "file_name": f"superclass_{uid}.png",
                "rel_dir": rel_path,
                "total_pixels": int(total),
            }
            
            # Calculate count and percentage for each superclass.
            for name, class_int in SUPER_CLASSES.items():
                count_val = int((merged == class_int).sum())
                pct_val = float(count_val / total) if total > 0 else 0.0
                
                rec[f"count_{name}"] = count_val
                rec[f"pct_{name}"] = pct_val
            
            new_records.append(rec)

print(f"Processed {len(new_records)} new masks.")

## SAVE METRICS

In [ ]:
if new_records:
    df_new = pd.DataFrame(new_records)
    
    # Combine with existing records if any.
    if records:
        df_existing = pd.DataFrame(records)
        df_out = pd.concat([df_existing, df_new], ignore_index = True)
        df_out = df_out.drop_duplicates(subset = "uid", keep = "last")
    else:
        df_out = df_new
    
    # Save to CSV.
    df_out.to_csv(OUT_CSV, index = False)
    print(f"Saved metrics to {OUT_CSV} ({len(df_out)} rows).")
else:
    if OUT_CSV.exists():
        print("No new masks found. Existing metrics unchanged.")
    else:
        print("No masks found and no metrics file created.")

## SUMMARY STATISTICS

In [ ]:
# Load final metrics.
if OUT_CSV.exists():
    df_metrics = pd.read_csv(OUT_CSV)
    
    print("SUPERCLASS METRICS SUMMARY")
    print(f"Total images: {len(df_metrics)}")
    print("")
    print("Mean percentages per superclass:")
    
    for name in SUPER_CLASSES.keys():
        col = f"pct_{name}"
        if col in df_metrics.columns:
            mean_pct = df_metrics[col].mean() * 100
            print(f"{name:20s}: {mean_pct:5.2f}%")
else:
    print("No metrics file found.")

## VISUALIZE SAMPLE RESULTS

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Define color map for superclasses.
colors = [
    "gray",       # other.
    "green",      # vegetation.
    "lightblue",  # sky.
    "brown",      # building.
    "black",      # pavement/road.
    "blue",       # water.
    "orange",     # vehicle/clutter.
]
cmap = ListedColormap(colors)

# Find sample images.
sample_superclass = list(IMAGE_DIR.rglob("superclass/superclass_*.png"))

if sample_superclass:
    sample_path = sample_superclass[0]
    uid = sample_path.stem.replace("superclass_", "")
    
    # Get corresponding original and segmented.
    parent_dir = sample_path.parent.parent
    original_path = parent_dir / "original" / f"gsv_{uid}.jpg"
    segmented_path = parent_dir / "segmented" / f"class_{uid}.png"
    
    fig, axes = plt.subplots(1, 3, figsize = (18, 6))
    
    # Original image.
    if original_path.exists():
        original = Image.open(original_path)
        axes[0].imshow(original)
        axes[0].set_title(f"Original: gsv_{uid}.jpg")
    axes[0].axis("off")
    
    # Segmented mask.
    if segmented_path.exists():
        segmented = np.array(Image.open(segmented_path))
        axes[1].imshow(segmented, cmap = "tab20")
        axes[1].set_title(f"Segmented: class_{uid}.png")
    axes[1].axis("off")
    
    # Superclass mask.
    superclass = np.array(Image.open(sample_path))
    axes[2].imshow(superclass, cmap = cmap, vmin = 0, vmax = 6)
    axes[2].set_title(f"Superclass: superclass_{uid}.png")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    # Legend.
    print("Superclass Legend:")
    for name, label in SUPER_CLASSES.items():
        print(f"{label}: {name} ({colors[label]})")
else:
    print("No superclass masks found for visualization.")